# RLSS2026 - PG Tutorial: Learning to Race With Policy Gradient

Website: https://rlsummerschool.com/

Github repository: https://github.com/araffin/rlss26-pg-tutorial

Slides: https://araffin.github.io/slides/rlss26-bbo-pg/

Gymnasium documentation: https://gymnasium.farama.org/

## Introduction

In this notebook, you will implement a policy gradient (PG) algorithm (REINFORCE) with a linear policy.
You will first work with a discretized action space (fixed number of actions), then extend to continuous control — building on the BBO concepts from the previous notebook.


In [ ]:
# for autoformatting
# !pip install jupyter-black
%load_ext jupyter_black

# for developping
%load_ext autoreload
%autoreload 2

## Install Dependencies

In [ ]:
import os

ON_COLAB = os.environ.get("COLAB_NOTEBOOK_ID") is not None

if ON_COLAB:
    !pip install git+https://github.com/araffin/rlss26-pg-tutorial --upgrade

In [ ]:
if ON_COLAB:
    !apt-get install ffmpeg  # For visualization

## Racing Environment

We will at first re-use the environment from the previous notebook and then use a slightly more complex version that is tailored for racing.

The main differences will be in the observation space and action space.

### Observation space

Instead of just using the lateral error and previous lateral error (like for the PD controller), the observation includes more information including the heading error, forward velocity and some information about the track to anticipate upcoming turns.

```python
# Observation: [lateral_error, heading_error, forward_velocity,
#               angular_velocity, left_wheel_speed, right_wheel_speed,
#               curvature, lookahead_lat_2, lookahead_lat_4, lookahead_lat_6]
```

### Action space

Instead of having a constant speed and handling only the steering, we will now control each wheel (left and right) indepedently (the action range is still normalized in $[-1, 1]$).

In the first part of this notebook, we will discretize the action space to make it simpler.

### Reward

Instead of the reward that was tailored to follow the line:
```python
reward = alive_bonus + lateral_penalty + 0.5 * heading_penalty + forward_reward
```

we will use a reward that encourages the robot to finish the track as fast as possible:

```python
# going fast close to the center of lane: (1 - abs(lateral_error)) * forward_velocity (normalized)
reward = (1.0 - (math.fabs(lateral_error) / self.off_track_threshold) ** 2) * (forward_velocity / max_speed)
```

### Vehicle dynamics

To make the environment slighly more challenging, it includes some drift / tyre-slip physics.

In [ ]:
import gymnasium as gym
import numpy as np

# register env
import pg_tutorial

# The discretizer
from pg_tutorial.wrappers import DiscreteActionWrapper

# Available tracks: "oval`, "s_track", "rounded_l", "hairpin", "custom".
env = gym.make("LineFollowerRacingCustom-v0", track_name="s_track")
# Discretize: 5 bins per action dimension, here we will have 5x5=25 actions
discrete_action_env = DiscreteActionWrapper(env, n_bins=5)

In [ ]:
print(f"{env.observation_space.shape=}")

In [ ]:
# Box(2) means that there is two continuous action: left and right wheel
print(f"{env.action_space=}")

In [ ]:
# The discretized version
print(f"{discrete_action_env.action_space=}")
# you can print the action map:
# discrete_action_env.action_map

## Policy Gradient with Discrete Actions

We will start with a simple linear policy (just a matrix).

### 1. Exercise (5 minutes): create the policy

A stochastic policy for discrete actions is parameterized by a neural network that outputs
raw scores (logits $z$), converted to probabilities via the softmax function:

$$
\pi_\theta(a \mid s) = \frac{\exp(z_a)}{\sum_{a'} \exp(z_{a'})} \quad \text{where} \quad z = W \cdot s + b
$$

The policy has two main methods:
- `get_action` to sample an action given an observation
- `get_log_prob` to compute the log probability of taking a given action (needed for the gradient)

**HINT**: We will use PyTorch `Categorical` distribution class to sample and compute the log probability

**HINT**: The policy architecture should be a `nn.Linear()` layer that maps each observation to a vector of raw scores (logits), one per action.

Note: if you want to test your implementation, there are some tests in the next cell

In [ ]:
import torch as th
import torch.nn as nn
from torch.distributions import Categorical


class DiscreteLinearPolicy(nn.Module):
    """
    A simple PyTorch model to represent a Linear policy
    in the discrete action setting.

    :param obs_dim: Dimension of the observation space (we assume that the observation is a 1D vector)
    :param action_dim: Dimension of the action space (here the number of discrete actions)
    """

    def __init__(self, obs_dim: int = 2, action_dim: int = 2) -> None:
        super().__init__()
        ### YOUR CODE HERE
        # You need to define the policy architecture (here just a `Linear()` layer)
        self.net = nn.Linear(obs_dim, action_dim, bias=True)
        ### END OF YOUR CODE

    def get_action(self, observation: th.Tensor, deterministic: bool = False) -> th.Tensor:
        ### YOUR CODE HERE
        # You need to define the policy action distribution for the current observation
        # and then either sample from it or return the most likely action (deterministic=True)
        #
        # TODO:
        # 1. Compute the logits (unnormalized probablity) for the given observation
        # 2. Create the action distribution object (using PyTorch `Categorical(...)`) for the given logits
        # 3. Either sample an action or return the most likely action depending on the value of deterministic

        # Compute the logits by doing a forward pass (here just a matrix multiplication behind the hood)
        # logits are un-normalized probabilities of taking each action
        # Here, this is the same as logits = weights @ observations (matrix multiplication)
        logits = self.net(observation)
        # Create the action distribution (using pytorch Categorical(logits=...))
        # A convenience class to sample, compute probabilties and find the argmax
        action_dist = Categorical(logits=logits)
        # Return the most likely action in case deterministic=True
        if deterministic:
            # Same as th.argmax(action_dist, dim=-1), get the most likely action
            return action_dist.mode
        # Otherwise sample from the action distribution and return the sampled action
        return action_dist.sample()
        ### END OF YOUR CODE

    def forward(self, observation: th.Tensor) -> th.Tensor:
        return self.get_action(observation)

    def get_log_prob(self, observations: th.Tensor, actions: th.Tensor) -> th.Tensor:
        ### YOUR CODE HERE
        # This is similar to `get_action()` but we compute different quantities
        #
        # TODO:
        # 1. Compute the logits (unnormalized probablity) for the given observation
        # 2. Create the action distribution object (using PyTorch `Categorical(...)`) for the given logits
        # 3. Return the log probability of taking the given actions (Note: Categorical has a `log_prob()` method)
        logits = self.net(observations)
        action_dist = Categorical(logits=logits)
        return action_dist.log_prob(actions)
        ### END OF YOUR CODE

Let's try the policy:

In [ ]:
# Setup
obs_dim = 2
action_dim = 2
policy = DiscreteLinearPolicy(obs_dim=obs_dim, action_dim=action_dim)

# Manually set weights to a known state:
# Weight matrix W = [[1.0, 0.0],
#                   [0.0, 1.0]]
# So logits = observation @ W.T
with th.no_grad():
    policy.net.weight.copy_(th.eye(action_dim, obs_dim))
    # deactivate bias
    if policy.net.bias is not None:
        policy.net.bias.copy_(th.zeros(action_dim))

# Sanity checks:
# 1. Test deterministic action selection
# Observation [1, 0] -> logits [1, 0] -> Action 0 is most likely
assert policy.get_action(th.tensor([1.0, 0.0]), deterministic=True) == 0
# Observation [0, 1] -> logits [0, 1] -> Action 1 is most likely
assert policy.get_action(th.tensor([0.0, 1.0]), deterministic=True) == 1
# Observation [-1, -1] -> logits [-1, -1] -> Tie (usually returns index 0)
assert policy.get_action(th.tensor([-1.0, -1.0]), deterministic=True) == 0

# 2. Test log-probabilities
# For obs [1, 0], logits are [1, 0].
# Prob(action 0) = exp(1) / (exp(1) + exp(0))
# LogProb(action 0) = 1 - log(exp(1) + exp(0))
obs = th.tensor([1.0, 0.0])
action = th.tensor(0)
expected_log_prob = 1.0 - th.log(th.exp(th.tensor(1.0)) + th.exp(th.tensor(0.0)))
assert th.allclose(policy.get_log_prob(obs, action), expected_log_prob)

# 3. Test batch processing and shapes
batch_size = 5
batch_obs = th.randn(batch_size, obs_dim)
batch_actions = th.randint(0, action_dim, (batch_size,))
log_probs = policy.get_log_prob(batch_obs, batch_actions)
assert log_probs.shape == (batch_size,)

actions = policy.get_action(batch_obs, deterministic=True)
assert actions.shape == (batch_size,)

# Different dimensions
obs_dim = 3
action_dim = 5
policy = DiscreteLinearPolicy(obs_dim=obs_dim, action_dim=action_dim)
assert policy.get_action(th.ones(obs_dim), deterministic=True).shape == ()

#### Evaluate and record a video

To evaluate and record a video, we will use the `evaluate_policy` helper that takes the policy and env as input and returns the `best_lap_time` (can be `inf`) and the mean episodic return.

```python
def evaluate_policy(
    policy: DiscreteLinearPolicy,
    env: gym.Env,
    n_eval_episodes: int = 5,
    video_name: str | None = None,
    video_save_path: str | None = None,
    verbose: bool = True,
    deterministic: bool = True,
) -> tuple[float, float]:
    """
    Evaluate a policy on an environment over multiple episodes.

    :param policy: A policy that takes observation as input and returns an action.
    :param env: The Gymnasium environment to evaluate on.
    :param n_eval_episodes: Number of episodes to evaluate.
    :param video_name: Optional name for recording a video of the evaluation.
    :param video_save_path: Optional directory path to save videos. If None, defaults to the parent
        directory of this file's folder.
    :param verbose: Whether to print evaluation results.
    :param deterministic: Wether to sample or take the most likely action
    :return: A tuple containing (best_lap_time, mean_episode_return).
    """
```

In [ ]:
from pathlib import Path

import pg_tutorial
from pg_tutorial.notebook_utils import show_videos
from pg_tutorial.notebook_utils import evaluate_policy

video_save_path = str(Path(pg_tutorial.__file__).parent.parent / "logs" / "videos")

# Available tracks: "oval`, "s_track", "rounded_l", "hairpin", "custom".
eval_env = DiscreteActionWrapper(
    gym.make(
        "LineFollowerRacingCustom-v0",
        track_name="s_track",
        render_mode="rgb_array",
        max_episode_steps=800,
    ),
    n_bins=5,
)
video_name = "discrete_random_linear_policy"
n_eval_episodes = 1

obs_dim = int(np.prod(eval_env.observation_space.shape))
n_actions = int(eval_env.action_space.n)

policy = DiscreteLinearPolicy(obs_dim, n_actions)

evaluate_policy(policy, eval_env, n_eval_episodes, video_name=video_name)
show_videos(video_save_path, prefix=video_name)

### 2. Exercise (10 minutes): write the data collection for one episode

The first step in policy gradient is to collect data by running the stochastic policy in the environment.

We will collect observations, taken actions and rewards for one episode.

**HINT**: Gymnasium `reset()` returns a tuple `(observation, info)` and `step()` returns `(observation, reward, terminated, truncated, info)`. Unpack with comma-separated variables, or use `_` to discard unused values.

**HINT**: `policy.get_action()` expects a `torch.Tensor` as input (you can use `th.as_tensor()` to convert a numpy array to a torch tensor)

**HINT**: For discrete actions, convert the action to an integer using `int()` or `.item()` before calling `env.step()`.

Note: if you want to test your implementation, there are some tests in the next cell

In [ ]:
def collect_one_episode(
    policy: DiscreteLinearPolicy,
    env: gym.Env,
    iteration: int = 1,
    seed: int = 0,
) -> tuple[th.Tensor, th.Tensor, np.ndarray, dict]:
    # Initialize buffers for one episode
    observations: list[th.Tensor] = []
    actions: list[th.Tensor] = []
    rewards: list[float] = []

    # Reset the env to start a new episode
    # Only seed for the very first episode
    current_obs, _ = env.reset(seed=seed if iteration == 1 else None)

    ### YOUR CODE HERE

    # TODO:
    # 1. Sample an action according to the current policy and observation
    # 2. Step in the env using the sampled action
    # 3. Retrieve the new transition data (observation, reward, ...)
    #  and update the numpy arrays (buffers)

    done = False

    while not done:
        # Sample action with current policy
        # Don't forget to first convert the observation to a torch Tensor using th.as_tensor()
        obs_tensor = th.as_tensor(current_obs)
        action = policy.get_action(obs_tensor)

        # Store transitions (observations and actions)
        observations.append(obs_tensor)
        actions.append(action)

        # Step in the env and retrieve new observation, reward, ...
        # Don't forget to convert the action to an integer
        next_obs, reward, terminated, truncated, info = env.step(int(action))
        # Check if the episode is over
        done = terminated or truncated

        # Store the reward
        rewards.append(float(reward))

        # Update current obs
        current_obs = next_obs

    ### END OF YOUR CODE

    # Convert lists to tensors
    obs_tensor = th.stack(observations)
    actions_tensor = th.stack(actions)

    # Return the collected data (info from the last step contains episode-ending metrics)
    return obs_tensor, actions_tensor, np.array(rewards), info

You can test the data collection with the following sanity checks:

In [ ]:
eval_env = DiscreteActionWrapper(gym.make("LineFollowerRacingCustom-v0"), n_bins=5)

obs_dim = int(np.prod(eval_env.observation_space.shape))
n_actions = int(eval_env.action_space.n)

# Deterministic test
seed = 8
th.random.manual_seed(seed)

policy = DiscreteLinearPolicy(obs_dim, n_actions)
obs_tensor, actions_tensor, rewards, _ = collect_one_episode(policy, eval_env, seed=seed)

n_steps = len(obs_tensor)
assert obs_tensor.shape == (n_steps, obs_dim)
assert actions_tensor.shape == (n_steps,)
assert rewards.shape == (n_steps,)
assert isinstance(obs_tensor, th.Tensor)

### 3. Exercise (5 minutes): compute the discounted episodic return

We will compute the discounted return $G_t = \sum_{k=0}^{T} \gamma^k r_{t+k}$ for each timestep of the episode (and for one episode only).
The most efficient way is to iterate **backwards** through the rewards and accumulate the return.

**HINT**: start from $G_T = 0$ and compute $G_{t} = r_t + \gamma \cdot G_{t+1}$.

**HINT**: to compute the discounted return $G_{t}$, you should use a special variable to accumulate it (`episode_return = reward[t] + gamma * episode_return`, here `episode_return` is the accumulator)


Note: if you want to test your implementation, there are some tests in the next cell

In [75]:
def compute_discounted_return(rewards: np.ndarray, gamma: float) -> th.Tensor:
    """
    Compute the discounted episodic return for one episode.

    :param rewards: a np array containing the immediate rewards (one per step)
    :param gamma: the discount factor (gamma=1.0 means undiscounted return)
    :return: the discounted return for each step
    """
    discounted_returns = th.zeros(len(rewards))
    ### YOUR CODE HERE

    # TODO:
    # 1. Iterate backwards through the rewards (from last to first)
    # 2. Accumulate the discounted return
    # 3. Store the result in `discounted_returns`
    current_return = 0.0
    for step in reversed(range(len(rewards))):
        current_return = rewards[step] + gamma * current_return
        discounted_returns[step] = current_return

    ### END OF YOUR CODE
    return discounted_returns

In [74]:
# Test discounted return
returns = compute_discounted_return(np.array([1.0, 2.0, 3.0]), gamma=0.9)
# G_0 = 1.0 + 0.9 * (2.0 + 0.9 * 3.0) = 1.0 + 0.9 * 4.7 = 5.23
# G_1 = 2.0 + 0.9 * 3.0 = 4.7
# G_2 = 3.0
assert th.allclose(returns, th.tensor([5.23, 4.7, 3.0]))

# Gamma = 1.0 (undiscounted): cumulative sum from the end
returns_flat = compute_discounted_return(np.array([1.0, 2.0, 3.0]), gamma=1.0)
assert th.allclose(returns_flat, th.tensor([6.0, 5.0, 3.0]))

# Single reward
returns_one = compute_discounted_return(np.array([42.0]), gamma=0.99)
assert th.allclose(returns_one, th.tensor([42.0]))

# Correct shape and type
r = np.array([1.0, 2.0])
assert compute_discounted_return(r, 0.9).shape == (2,)
assert isinstance(compute_discounted_return(r, 0.9), th.Tensor)

### 4. Exercise (10 minutes): write the policy update

The policy gradient theorem (REINFORCE) states that the gradient of the expected return is:

$$\nabla_\theta J(\theta) = \mathbb{E}_{\tau \sim \pi_\theta} \left[ \nabla_\theta \log \pi_\theta(a_t \mid s_t) \cdot G_t \right]$$

In practice, we approximate the expectation with a single episode sample and use the discounted return as the advantage:

$$\mathcal{L}_{PG} = - \frac{1}{T} \sum_{t=0}^{T-1} \log \pi_\theta(a_t \mid s_t) \cdot G_t$$

The negative sign turns the gradient ascent objective into a minimization loss (PyTorch default).

Now we will combine the previous exercises together into a training loop:
- Instantiate `DiscreteLinearPolicy` and an optimizer (you can use `Adam`)
- In each iteration, collect one episode, compute discounted returns, compute the policy gradient loss, and update the policy

**HINT**: The optimizer can be created using `th.optim.Adam(policy.parameters(), lr=learning_rate)`

**HINT**: The policy gradient loss should contain the term-wise multiplication `advantages * log_probs` and is a scalar (you will need to use `.mean()` or `.sum()`

Note: run the training and evaluation cells below to verify your implementation.

In [76]:
from collections import deque
import warnings
import time

from tqdm.notebook import tqdm


def run_policy_gradient(
    env: gym.Env,
    gamma: float = 0.99,
    learning_rate: float = 1e-2,
    n_iterations: int = 20,
    seed: int = 0,
    smoothing_window: int = 50,
    log_freq: int = 5,
) -> DiscreteLinearPolicy:
    # Env info
    obs_dim = int(np.prod(env.observation_space.shape))
    n_actions = int(env.action_space.n)
    total_timesteps = 0

    # Pseudo-random generator seeding for reproducible results
    np.random.seed(seed)
    th.manual_seed(seed)

    ### YOUR CODE HERE
    # TODO:
    # 1. Instantiate a DiscreteLinearPolicy object
    # 2. Create an Adam optimizer for policy.parameters() with the given learning_rate

    # Instantiate the policy
    policy = DiscreteLinearPolicy(obs_dim, n_actions)

    # Create the optimizer
    optimizer = th.optim.Adam(policy.parameters(), lr=learning_rate)
    ### END OF YOUR CODE

    # Report some statistics, mean over last episodes
    episode_returns: deque[float] = deque(maxlen=smoothing_window)
    episode_lengths: deque[int] = deque(maxlen=smoothing_window)
    n_episodes = 0
    start_time = time.monotonic()
    best_lap_time = last_lap_time = 0.0

    for iteration in tqdm(range(1, n_iterations + 1)):

        ### YOUR CODE HERE
        # TODO:
        # 1. Collect one episode using `collect_one_episode(...)
        # 2. Compute discounted returns using `compute_discounted_return(...)`
        # 3. Compute the log probability of the taken actions using `policy.get_log_prob()`
        # 5. Compute the policy gradient loss (be careful with the sign!)

        # Collect data for one episode
        obs_tensor, actions_tensor, rewards, info = collect_one_episode(policy, env, iteration, seed=seed)
        # For stats
        total_timesteps += len(rewards)

        # Compute discounted returns for one episode
        discounted_returns = compute_discounted_return(rewards, gamma)

        # Compute advantages (here with no baseline, it is the same as the discounted reward)
        advantages = discounted_returns

        # Update the policy with policy gradient loss
        # Remember that we are doing gradient ascent (careful with the sign)
        log_probs = policy.get_log_prob(obs_tensor, actions_tensor)
        pg_loss = -(advantages * log_probs).mean()

        ### END OF YOUR CODE

        # Backpropagate and update
        optimizer.zero_grad()
        pg_loss.backward()
        optimizer.step()

        # Log racing metrics if available
        if "best_lap_time" in info:
            best_lap_time = info["best_lap_time"]
        if "last_lap_time" in info:
            last_lap_time = info["last_lap_time"]

        n_episodes += 1
        episode_lengths.append(len(rewards))
        episode_returns.append(sum(rewards))

        # Logging
        if (iteration % log_freq) == 0:
            print(f" {iteration=}/{n_iterations} ".center(30, "="))
            time_elapsed = time.monotonic() - start_time
            fps = total_timesteps / time_elapsed
            # Log racing metrics if available
            if best_lap_time > 0.0:
                print(f"racing/best_lap_time: {best_lap_time:.2f}s")
                print(f"racing/last_lap_time: {last_lap_time:.2f}s")
            print(f"rollout/{n_episodes=}")
            print(f"rollout/{np.mean(episode_returns)=:.2f} +/- {np.std(episode_returns):.2f}")
            print(f"rollout/{np.mean(episode_lengths)=:.2f} +/- {np.std(episode_lengths):.2f}")
            print(f"time/{total_timesteps=}")
            print(f"time/{time_elapsed=:.0f}")
            print(f"time/{fps=:.2f}")
            print(f"train/{pg_loss=:.4f}")
            print("=" * 30)
    return policy

#### Evaluate and record a video of discrete policy

Note: you can make the evaluation faster by passing `video_name=None`

In [ ]:
# Available tracks: "oval`, "s_track", "rounded_l", "hairpin", "custom".
env = DiscreteActionWrapper(
    gym.make(
        "LineFollowerRacingCustom-v0",
        track_name="s_track",
        render_mode="rgb_array",
        max_episode_steps=800,
    ),
    n_bins=5,
)
video_name = "discrete_linear_policy"
n_eval_episodes = 1

# Normalize obs to have mean ~ 0.0, std ~ 1.0
env = gym.wrappers.NormalizeObservation(env)

policy = run_policy_gradient(
    env,
    gamma=0.99,
    learning_rate=1e-2,
    n_iterations=500,
    seed=8,
)

In [ ]:
evaluate_policy(policy, env, n_eval_episodes, video_name=video_name, deterministic=True)
show_videos(video_save_path, prefix=video_name)

## Policy Gradient with Continuous Actions


Discrete actions are a bit limiting, let's see how far we can go with continuous actions.
It will be very similar to the discrete action case.

### 1. Exercise (10 minutes): create the continuous policy

For continuous actions the policy outputs a mean action and uses a learnable log standard deviation
(`log_std`) to form a diagonally factored Gaussian (one independent Gaussian per action dimension):

$$\pi_\theta(\mathbf{a} \mid \mathbf{s}) = \prod_{i=1}^{\mathcal{A}} \mathcal{N}(a_i; \, \mu_i(\mathbf{s}), \, \sigma^2)$$

where the mean $\boldsymbol{\mu}(\mathbf{s}) = W \cdot \mathbf{s} + b$ is a linear function of the observation,
and $\sigma = \exp(\log_\sigma)$ is kept positive via the log-parameterization.

The log-probability decomposes across independent action dimensions:

$$\log \pi_\theta(\mathbf{a} \mid \mathbf{s}) = \sum_{i=1}^{\mathcal{A}} \log \mathcal{N}(a_i; \, \mu_i, \, \sigma^2)$$

where $\mathcal{N}$ is the probability density function for one Gaussian/Normal distribution.

The policy has two main methods:
- `get_action` to compute the action to take given an observation
- `get_log_prob` to compute the log probability of taking a given action for a given observation

**HINT**: Use PyTorch `Normal` distribution class to sample and compute the log probability. The log probabilities across action dimensions should be summed (independent dimensions, you can use `.sum(dim=-1)` for that).

**HINT**: The `log_std` is a learnable parameter initialized to `log(std_init)`. We use the log-parameterization to guarantee that the standard deviation stays positive.

Note: if you want to test your implementation, there are some tests in the next cell

In [ ]:
import torch as th
import torch.nn as nn
from torch.distributions import Normal


class ContinuousLinearPolicy(nn.Module):
    """
    A simple PyTorch model to represent a Linear policy
    in the continuous action setting.

    :param obs_dim: Dimension of the observation space (we assume that the observation is a 1D vector)
    :param action_dim: Dimension of the action space
    :param std_init: Initial standard deviation
    """

    def __init__(self, obs_dim: int = 2, action_dim: int = 2, std_init: float = 1.0) -> None:
        super().__init__()
        ### YOUR CODE HERE
        # You need to define the policy architecture (here just a `Linear()` layer)
        # you can replace it later with something more complex
        self.net = nn.Linear(obs_dim, action_dim, bias=True)
        # self.net = nn.Sequential(nn.Linear(obs_dim, 16), nn.ReLU(), layer_init(nn.Linear(16, action_dim)))
        # State-Independent log standard deviation
        # We use the log to make sure std > 0
        # Note: we could make it state-dependent by having another network
        # that outputs the std (self.net = Linear(obs_dim, action_dim * 2))
        self.log_std = nn.Parameter(th.ones(action_dim) * np.log(std_init))

        ### END OF YOUR CODE

    def get_action(self, observation: th.Tensor, deterministic: bool = False) -> th.Tensor:
        ### YOUR CODE HERE
        # You need to define the policy action distribution for the current observation
        # and then either sample from it or return the most likely action (deterministic=True)
        #
        # TODO:
        # 1. Compute the mean action for the given observation
        # 2. Create the action distribution object (using PyTorch `Normal(mean, std)`)
        # 3. Either sample an action or return the most likely action depending on the value of deterministic

        # Compute the mean by doing a forward pass (here just a matrix multiplication behind the hood)
        # mean is the expected action under the Normal distribution
        # Here, this is the same as action_mean = weights @ observations + bias (matrix multiplication)
        action_mean = self.net(observation)
        # Compute the standard deviation (you will need to use `exp()`)
        # it must have the same shape as `action_mean`
        action_std = th.ones_like(action_mean) * self.log_std.exp()

        # Create the action distribution (using pytorch Normal)
        # A convenience class to sample, compute probabilties and find the mean (mode)
        action_dist = Normal(action_mean, action_std)
        # Return the most likely action in case deterministic=True
        # A convenience class to sample, compute probabilties and find the best action
        if deterministic:
            # Get the most likely action (here same as action_mean)
            return action_dist.mode
        # Otherwise sample from the action distribution and return the sampled action
        return action_dist.sample()
        ### END OF YOUR CODE

    def forward(self, observation: th.Tensor) -> th.Tensor:
        return self.get_action(observation)

    def get_log_prob(self, observations: th.Tensor, actions: th.Tensor) -> th.Tensor:
        ### YOUR CODE HERE
        # This is similar to `get_action()` but we compute different quantities
        #
        # TODO:
        # 1. Compute the action distribution parameters (mean, std) for the given observation
        # 2. Create the action distribution object (using PyTorch `Normal(...)`)
        # 3. Return the log probability of taking the given actions (Note: Normal has a `log_prob()` method)
        action_mean = self.net(observations)
        action_std = th.ones_like(action_mean) * self.log_std.exp()
        action_dist = Normal(action_mean, action_std)
        # Sum all action dimensions (treating them as independent, you will need to use sum(dim=1))
        return action_dist.log_prob(actions).sum(dim=1)
        ### END OF YOUR CODE

Let's test the continuous policy with some sanity checks (deterministic = mean, shapes, log-prob ordering):

In [ ]:
# Test continuous linear policy
obs_dim = 3
action_dim = 2
policy = ContinuousLinearPolicy(obs_dim=obs_dim, action_dim=action_dim, std_init=1.0)

# 1. Deterministic action equals mean
obs = th.zeros(obs_dim)
with th.no_grad():
    policy.net.weight.copy_(th.zeros(action_dim, obs_dim))
    policy.net.bias.copy_(th.tensor([0.5, -0.3]))
assert th.allclose(policy.get_action(obs, deterministic=True), th.tensor([0.5, -0.3]))

# 2. Random sampling returns correct shape
action = policy.get_action(th.randn(obs_dim), deterministic=False)
assert action.shape == (action_dim,)

# 3. Batched actions and log probs have correct shapes
batch_size = 4
batch_obs = th.randn(batch_size, obs_dim)
actions = policy.get_action(batch_obs, deterministic=True)
assert actions.shape == (batch_size, action_dim)

batch_actions = th.randn(batch_size, action_dim)
log_probs = policy.get_log_prob(batch_obs, batch_actions)
assert log_probs.shape == (batch_size,)

# 4. Log prob of the mean should be the highest per-dimension
with th.no_grad():
    test_obs = th.tensor([1.0, 2.0, 3.0])
    mean_action = policy.get_action(test_obs, deterministic=True)
    log_prob_mean = policy.get_log_prob(test_obs, mean_action)
    log_prob_other = policy.get_log_prob(test_obs, th.tensor([99.0, -99.0]))
    assert log_prob_mean > log_prob_other

### 2. Exercise (4 minutes): write the data collection for one episode (continuous actions)

Same structure as `collect_one_episode` in the discrete case, but with two important differences.

The Gaussian policy has unbounded support ($\mathbb{R}^\mathcal{A}$), so actions sent to the env must be clipped to the valid range:

$$\mathbf{a\_env}_t \leftarrow \text{clip}(\mathbf{a}_t, \, \mathbf{a}_{\text{low}}, \, \mathbf{a}_{\text{high}})$$

However, we store the unbounded action in the buffer for the policy update.

**HINT**: The Gaussian policy can sample actions outside `[-1, 1]`. Clip with `np.clip()` to match the `env.action_space` limits.

**HINT**: The env is not wrapped in `DiscreteActionWrapper` here, so `env.action_space` is a `Box` (not `Discrete`). Actions are NumPy arrays, not integers.

Note: if you want to test your implementation, there are some tests in the next cell

In [ ]:
def collect_one_episode_continuous(
    policy: ContinuousLinearPolicy,
    env: gym.Env,
    iteration: int = 1,
    seed: int = 0,
) -> tuple[th.Tensor, th.Tensor, np.ndarray, dict]:
    # Initialize buffers for one episode
    observations: list[th.Tensor] = []
    actions: list[th.Tensor] = []
    rewards: list[float] = []

    # Reset the env to start a new episode
    # Only seed for the very first episode
    current_obs, _ = env.reset(seed=seed if iteration == 1 else None)

    done = False

    while not done:
        ### YOUR CODE HERE

        # TODO:
        # 1. Sample an action with the current policy `policy.get_action(...)`
        # 3. Convert to NumPy and clip the action to the action space boundary
        # 4. Step in the environment using the clipped action
        
        # Sample action with current policy
        # Don't forget to first convert the observation to a torch Tensor using th.as_tensor()
        obs_tensor = th.as_tensor(current_obs)
        action = policy.get_action(obs_tensor)

        # Store transitions (observations and actions)
        observations.append(obs_tensor)
        actions.append(action)

        # Convert from Torch Tensor to NumPy array
        action_np = action.numpy()
        # Clip infinite support Gaussian to correct bounds
        action_np = np.clip(action_np, env.action_space.low, env.action_space.high)

        # Step in the env and retrieve new observation, reward, ...
        next_obs, reward, terminated, truncated, info = env.step(action_np)
        
        ### END OF YOUR CODE
        
        # Check if the episode is over
        done = terminated or truncated

        # Store the reward
        rewards.append(float(reward))

        # Update current obs
        current_obs = next_obs

    # Convert lists to tensors
    obs_tensor = th.stack(observations)
    actions_tensor = th.stack(actions)

    # Return the collected data (info from the last step contains episode-ending metrics)
    return obs_tensor, actions_tensor, np.array(rewards), info

You can test the continuous episode collection with the following sanity checks (shapes, types, and info dict):

In [ ]:
# Test data collection for continuous control
env = gym.make("LineFollowerRacingCustom-v0")

obs_dim = int(np.prod(env.observation_space.shape))
action_dim = int(np.prod(env.action_space.shape))

policy = ContinuousLinearPolicy(obs_dim, action_dim)
obs_tensor, actions_tensor, rewards, info = collect_one_episode_continuous(policy, env, seed=42)

n_steps = len(obs_tensor)
assert obs_tensor.shape == (n_steps, obs_dim), f"obs shape mismatch: {obs_tensor.shape}"
assert actions_tensor.shape == (n_steps, action_dim), f"action shape mismatch: {actions_tensor.shape}"
assert rewards.shape == (n_steps,), f"reward shape mismatch: {rewards.shape}"
assert isinstance(obs_tensor, th.Tensor)
assert isinstance(actions_tensor, th.Tensor)
assert isinstance(rewards, np.ndarray)

# All actions fed to the env were clipped — verify last info dict is not empty
assert len(info) > 0

### 3. Exercise (5 minutes): write the policy update (continuous actions)

Combine everything: episode collection (`collect_one_episode_continuous`), discounted return computation, and policy gradient update.
This is analogous to `run_policy_gradient` in the discrete case, but:
- Uses `ContinuousLinearPolicy` and `collect_one_episode_continuous`

A useful trick for stability is **advantage normalization**, which centers and scales the advantages:

$$\tilde{A}_t = \frac{A_t - \bar{A}}{\sigma_A} \approx \frac{A_t - \frac{1}{T} \sum_{k} A_k}{\sqrt{\frac{1}{T} \sum_{k} (A_k - \bar{A})^2 + \epsilon}}$$

This prevents any single timestep from dominating the gradient and can speed up convergence.

**HINT**: The policy gradient loss will be exactly the same as for the discrete actions

**Note**: You only need to write the data collection and loss computation. The policy instantiation (`ContinuousLinearPolicy`), optimizer (`th.optim.Adam`), backpropagation chain (`zero_grad` -> `backward` -> `step`), and logging are all provided outside your code block.

**HINT**: Try enabling the advantage normalization (substract its mean and/or divide by its std) and compare training stability with and without it.

Note: run the training and evaluation cells below to verify your implementation.

In [ ]:
def run_policy_gradient_continuous(
    env: gym.Env,
    gamma: float = 0.99,
    learning_rate: float = 1e-2,
    n_iterations: int = 20,
    seed: int = 0,
    smoothing_window: int = 50,
    log_freq: int = 5,
    policy: ContinuousLinearPolicy | None = None,
) -> ContinuousLinearPolicy:
    # Env info
    obs_dim = int(np.prod(env.observation_space.shape))
    action_dim = int(np.prod(env.action_space.shape))
    total_timesteps = 0

    # Pseudo-random generator seeding for reproducible results
    np.random.seed(seed)
    th.manual_seed(seed)

    # Instantiate the policy
    if policy is None:
        policy = ContinuousLinearPolicy(obs_dim, action_dim)

    # Create the optimizer
    optimizer = th.optim.Adam(policy.parameters(), lr=learning_rate)

    # Report some statistics, mean over last episodes
    episode_returns: deque[float] = deque(maxlen=smoothing_window)
    episode_lengths: deque[int] = deque(maxlen=smoothing_window)
    n_episodes = 0
    start_time = time.monotonic()
    best_lap_time = last_lap_time = 0.0

    for iteration in tqdm(range(1, n_iterations + 1)):

        ### YOUR CODE HERE
        # TODO:
        # 1. Collect one episode using collect_one_episode_continuous(...)
        # 2. Compute discounted returns using compute_discounted_return(...)
        # 3. Compute the log probabilities of the taken actions using policy.get_log_prob(...)
        # 5. Compute the policy gradient loss

        # Collect data for one episode
        obs_tensor, actions_tensor, rewards, info = collect_one_episode_continuous(policy, env, iteration, seed=seed)
        # For stats
        total_timesteps += len(rewards)

        # Compute discounted returns for one episode
        discounted_returns = compute_discounted_return(rewards, gamma)

        # Compute advantages (here with no baseline, it is the same as the discounted reward)
        advantages = discounted_returns
        # advantages = advantages - advantages.mean()

        # Update the policy with policy gradient loss
        log_probs = policy.get_log_prob(obs_tensor, actions_tensor)
        pg_loss = -(advantages * log_probs).mean()

        ### END OF YOUR CODE

        # Backpropagate and update
        optimizer.zero_grad()
        pg_loss.backward()
        optimizer.step()

        # Log racing metrics if available
        if "best_lap_time" in info:
            best_lap_time = info["best_lap_time"]
        if "last_lap_time" in info:
            last_lap_time = info["last_lap_time"]

        n_episodes += 1
        episode_lengths.append(len(rewards))
        episode_returns.append(sum(rewards))
        
        # Logging
        if (iteration % log_freq) == 0:
            print(f" {iteration=}/{n_iterations} ".center(30, "="))
            time_elapsed = time.monotonic() - start_time
            fps = total_timesteps / time_elapsed
            std = policy.log_std.exp().mean().item()
            # Log racing metrics if available
            if best_lap_time > 0.0:
                print(f"racing/best_lap_time: {best_lap_time:.2f}s")
                print(f"racing/last_lap_time: {last_lap_time:.2f}s")
            print(f"rollout/{n_episodes=}")
            print(f"rollout/{np.mean(episode_returns)=:.2f} +/- {np.std(episode_returns):.2f}")
            print(f"rollout/{np.mean(episode_lengths)=:.2f} +/- {np.std(episode_lengths):.2f}")
            print(f"time/{total_timesteps=}")
            print(f"time/{time_elapsed=:.0f}")
            print(f"time/{fps=:.2f}")
            print(f"train/{pg_loss=:.4f}")
            print(f"train/{std=:.2f}")
            print("=" * 30)

    return policy

In [ ]:
# Available tracks: "oval`, "s_track", "rounded_l", "hairpin", "custom".
env = gym.make(
    "LineFollowerRacingCustom-v0",
    track_name="s_track",
    render_mode="rgb_array",
    max_episode_steps=800,
)
video_name = "continuous_linear_policy"
n_eval_episodes = 1

# Normalize obs to have mean ~ 0.0, std ~ 1.0
env = gym.wrappers.NormalizeObservation(env)

# policy = ContinuousLinearPolicy(obs_dim=12, action_dim=2, std_init=1.0)

policy = run_policy_gradient_continuous(
    env,
    gamma=0.99,
    learning_rate=1e-2,
    n_iterations=500,
    seed=0,
    log_freq=10,
    # policy=policy,  # you can pass a
)

#### Evaluate and record a video of continuous policy

Note: you can make the evaluation faster by passing `video_name=None`

In [ ]:
evaluate_policy(policy, env, n_eval_episodes, video_name=video_name, deterministic=True)
show_videos(video_save_path, prefix=video_name)

### Going further

- play with the different hyperparameters
- change the policy network to have something more complex
- implement the "n_steps" variant of policy gradient (instead collecting data from one episode, collect `n_steps` of data)

## Conclusion

What we have seen in this notebook:
- implement the policy gradient algorithm for discrete actions
- implement the policy gradient algorithm for continuous actions